In [43]:
import ipy

In [2]:
import xtc.graphs.xtc.op as O
from xtc.graphs.xtc.graph import XTCGraph

def matmul_graph(I: int, J: int, K: int, dtype: str) -> XTCGraph:
   """Create a graph computing C = A @ B."""
   a = O.tensor((I, K), dtype, name="A")
   b = O.tensor((K, J), dtype, name="B")
   with O.graph(name="matmul") as gb:
     C = O.matmul(a, b, name="C")
   return gb.graph

I, J, K, dtype = 16, 1000, 512, "float32"
graph = matmul_graph(I=I,J=J,K=K,dtype=dtype)

print(graph)

graph:
  name: matmul
  inputs:
  - %3 : 16x512xfloat32
  - %4 : 512x1000xfloat32
  outputs:
  - %5 : 16x1000xfloat32
  nodes:
  - %5: matmul(%3, %4) {name = 'C'} : [16x512xfloat32, 512x1000xfloat32] -> [16x1000xfloat32]



In [44]:
ipy.display_nradio("print", "Print options", "none", "source", "transformed", "assembly")

RadioButtons(description='Print options', options=(('none', 'none'), ('source', 'source'), ('transformed', 'tr…

In [46]:
from xtc.backends.tvm import Backend
from xtc.runtimes.host import HostRuntime

# Problem setup
I, J, K, dtype = 16, 1000, 512, "float32"
graph = matmul_graph(I, J, K, dtype)
backend = Backend(graph)

# Compile (no transformations, just default loop structure)
scheduler = backend.get_scheduler()
schedule = scheduler.schedule()

print_opts = dict(
    print_source_ir=ipy.nradio_value("print", "source"),
    print_transformed_ir=ipy.nradio_value("print", "transformed"),
    print_assembly=ipy.nradio_value("print", "assembly"),
)
compiler = backend.get_compiler(dump_file="matmul_naive", shared_lib=True, **print_opts)
module = compiler.compile(schedule)

# Evaluate and display results
peak_flops = HostRuntime.get().evaluate_flops(dtype)
evaluator = module.get_evaluator()
results, _, _ = evaluator.evaluate()
perf = (I * J * K) / min(results) / peak_flops * 100

print(f"\nPeak perfromance: {perf:.2f} %")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def main(_72_: T.Buffer((16, 512), "float32"), _73_: T.Buffer((512, 1000), "float32"), C: T.Buffer((16, 1000), "float32")):
        T.func_attr({"from_legacy_te_schedule": T.bool(True), "tir.noalias": T.bool(True)})
        for i, j in T.grid(16, 1000):
            C_1 = T.Buffer((16000,), data=C.data)
            C_1[i * 1000 + j] = T.float32(0.0)
            for k in range(512):
                cse_var_1: T.int32 = i * 1000 + j
                _72__1 = T.Buffer((8192,), data=_72_.data)
                _73__1 = T.Buffer((512000,), data=_73_.data)
                C_1[cse_var_1] = C_1[cse_var_1] + _72__1[i * 512 + k] * _73__1[k * 1000 + j]

Peak perfromance: 1.74 %
